In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Run this cell if you're using from colab
#!git clone https://github.com/R-Oc-A/HackathonPastryLPV.git
#!wget https://github.com/R-Oc-A/HackathonPastryLPV/releases/download/IntensityGrids/grids.tar.gz
#!wget https://github.com/R-Oc-A/HackathonPastryLPV/releases/download/IntensityGrids/famias.tar.gz
#!tar -xvf grids.tar.gz
#!tar -xvf famias_grids.tar.gz
#!pip install https://github.com/R-Oc-A/HackathonPastryLPV/releases/download/wheel/pastrypy-010-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
#!pip install tomli_w
#import sys
#sys.path.append('/content/HackathonPastryLPV')
import os
#os.environ["GRIDS"]="/content/ema_parquets/"

In [ ]:
# try:
#     import google.colab  # noqa: F401
# except ImportError:
#     import pyvista as pv
# else:
#     !wget "https://fem-on-colab.github.io/releases/vtk-install.sh" -O "/tmp/vtk-install.sh" && bash "/tmp/vtk-install.sh"
#     import pyvista as pv

In [ ]:
import pastrypy as psp
import pulsation_description as plsd
import line_profile_description as lpd
import tomli_w
import polars as pl
import matplotlib.pyplot as plt
import healpy as hp
import numpy as np

In [ ]:
import pyvista as pv
from trame_pyvista.jupyter import launch_server

# Grid Familiarization activity

### Regular Colatitude Azimuth (RCA) grid

In [ ]:
nside=4
sphere = pv.Sphere(radius=1, phi_resolution=nside*2, theta_resolution=nside*4)
plotter = pv.Plotter(notebook=True)
plotter.add_mesh(sphere, show_edges=True, color='white', opacity=1)
plotter.add_points(np.column_stack([x, y, z]), color='red', point_size=1)
plotter.show(jupyter_backend="html", return_viewer=True)

### Healpix discretization
Hierarchical equal area pixelization of the surface

In [ ]:
nside = 4
npix = hp.nside2npix(nside)
boundaries = hp.boundaries(nside, np.arange(npix),step=1)
print(f"the number of surface cells is {npix}")

In [ ]:
all_vertices = np.zeros((npix * 4,3))
faces =[]
for i in range(npix):
    # Extract the 4 corners for this pixel
    corners = boundaries[i,:, :].T
    
    all_vertices[i*4:(i+1)*4]=corners
    # Define the face: 4 vertices in the 
    # face
    base = i*4
    faces.append([4, base, base+1, base+2, base+3])

faces_flat = np.hstack(faces)
mesh = pv.PolyData(all_vertices, faces=faces_flat)
centers = mesh.cell_centers()
plotter = pv.Plotter(notebook=True)
plotter.add_mesh(mesh, show_edges=True, color='white', edge_color='black', line_width=1)
plotter.add_mesh(centers,color = 'red', point_size=5, render_points_as_spheres=True)
plotter.add_mesh(pv.Sphere(radius=0.99),color = 'white', opacity = 0.3)
plotter.show(jupyter_backend="html", return_viewer=True)


Healpix enables easy manipulation of the surface

In [ ]:
nside = 64
npix = hp.nside2npix(nside)
np.random.seed(12)
signal = np.random.normal(size=npix)
hp.mollview(signal,title="signal")
hp.graticule()

In [ ]:
gal_cut = np.radians(10)
phase=+0.25
mask = np.zeros(npix,dtype = np.float32)
mask[hp.query_disc(nside,hp.ang2vec(np.pi/2,0+2*np.pi*phase),np.radians(30))] = 1
eclipse_mask = np.zeros(npix,dtype=np.float32)
apodized_mask = np.clip(hp.smoothing(mask,fwhm=np.radians(10)),0,None)
hp.mollview(apodized_mask, cmap='plasma')
hp.graticule()
plt.savefig(fname="molview_left.png")

In [ ]:
hp.mollview(signal,alpha=(1-apodized_mask),title="signal with apodized mask")

In [ ]:
hp.mollview(signal,alpha=(1-mask),title="signal with boolean mask")

### Triangularization 
This is based on the marching step triangularization algorithm

In [ ]:
sphere_points = psp.sphere_triangulation_points()
sphere_triangles = psp.sphere_triangulation_triangles()
sphere_triangles.head()
sphere_points.head()

In [ ]:
points_extracted = sphere_points[['x coordinate','y coordinate','z coordinate']].to_numpy()
triangles_extracted = sphere_triangles[['first vertex','second vertex','third vertex']].to_numpy()
faces = np.hstack([np.full((triangles_extracted.shape[0],1),3),triangles_extracted])
mesh = pv.PolyData(points_extracted,faces)
plotter = pv.Plotter(notebook=True)
plotter.add_mesh(mesh)
plotter.add_axes()
plotter.show(jupyter_backend="html", return_viewer=True)

In [ ]:
roche_points = psp.roche_deformed_triangulation_points(w=0.4,length=0.3)#w is the critical frequency, length is the length of each side of the triangle, approximately
roche_triangles = psp.roche_deformed_triangulation_triangles(w=0.4,length=0.3)

In [ ]:
points_extracted = sphere_points[['x coordinate','y coordinate','z coordinate']].to_numpy()
triangles_extracted = sphere_triangles[['first vertex','second vertex','third vertex']].to_numpy()
faces = np.hstack([np.full((triangles_extracted.shape[0],1),3),triangles_extracted])
mesh = pv.PolyData(points_extracted,faces)
plotter = pv.Plotter(notebook=True)
plotter.add_mesh(mesh)
plotter.add_axes()
plotter.show(jupyter_backend="html", return_viewer=True)

# Pulstar 
Here you specify the star you'll be modelling as well as the modes of pulsation
### Star Data

In [ ]:
star_data=plsd.StarData(mass=10.0,
                        radius=6.93,
                        effective_temperature=21642.0,
                        v_omega=20.0,
                        inclination_angle=45.0)

### Phases of pulsation to model (time in days)

In [ ]:
time_points=plsd.TimePoints(Uniform=plsd.UniformTime(start=0.0,end=0.2,step=0.01))
#time_points=plsd.TimePoints(Uniform=plsd.ExplicitTime([0.0,0.01,0.4]))

### Pulsation modes to model

In [ ]:
nonrot= plsd.NonRot()
pertcor=plsd.PerturbCor()
tar = plsd.TAR()
cendef = plsd.CenDef(coefficient_expansion=[-0.856,0.01,0.0])
rotation_regime = plsd.RotationRegime(NonRotating=None,PerturbativeCoriolis=None,Tar=None,CentrifugalDeformation=None)
rotation_regime.NonRotating=nonrot
#rotation_regime.PerturbativeCoriolis=pertcor
#rotation_regime.Tar=tar
#rotation_regime.CentrifugalDeformation=cendef

In [ ]:
mode_test=plsd.Mode(l=2,m=0,
                rel_dr=0.024,
                k=0.03,frequency=5.38,
                phase_offset=0.0,
                rel_dtemp=2.62,
                phase_rel_dtemp=180.0,
                rel_dg=10.0,
                phase_rel_dg=34.0,
                rotation_effects = rotation_regime)

### Mesh options

In [ ]:
#mesh=plsd.Mesh(Sphere=plsd.SphericalStar(theta_step=2.0,phi_step=4.0))
mesh=plsd.Mesh(HSphere=plsd.HealpixStar(depth=5))
#mesh = plsd.Mesh(TSphere=plsd.TriangulatedStar(triangle_length=0.3))
#mesh = plsd.Mesh(DSphere=plsd.DeformedStar(triangle_length=0.3,rotation_frequency=0.2))

### Putting it all together

In [ ]:
pulsconfig=plsd.PulstarConfig(
    mode_data=[mode],
    star_data=star_data,
    time_points=time_points,
    mesh=mesh)

pulsconfig_dict=pulsconfig.model_dump(exclude_none=True)
puls_toml_string=tomli_w.dumps(pulsconfig_dict)
print(puls_toml_string)

## First run

In [ ]:
pulse_df = psp.pulstar(puls_toml_string)
pulse_df.sort("time").head(5)

# Profile configuration
Here you specify the line profile variability you want to observe.

### FAMIAS approach
For this approach you select an absorption line from your observations and you model it as a Gaussian profile

In [ ]:
gaussian_config = lpd.GaussianProfile(
    sigma=0.5,
    eq_w=0.5,
    alpha_w=0.0,
    zero_point_shift=0.0,
    # In Angstrom units
    central_wavelength=4100.0,
    left_wavelength=4094.0,
    right_wavelength=4106.0,
    step=0.003125,
    t_eff=21000,
    mass = 10.0,
    radius = 6.93,
)

prof_config_dict=gaussian_config.model_dump(exclude_none=True)

prof_gauss_toml_string=tomli_w.dumps(prof_config_dict)
print(prof_gauss_toml_string)

### Model atmosphere interpolation
Here we use a precomputed grid of model atmospheres to estimate local quantities out of the pulstar simulation

#### Select the chunk of the spectrum you want to model

In [ ]:
wl_range=lpd.WavelengthRange(start=4551.0,end=4555.0,step=0.0033)

### Select the model atmospheres that you want to use

In [ ]:

path_to_grids = os.getenv("GRIDS")
print(path_to_grids)

grid1=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=20000.0,log_gravity=3.5,metalicity=0.0,filename="lp0000_20000_0350_0020.parquet"))
grid2=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=20000.0,log_gravity=3.8,metalicity=0.0,filename="lp0000_20000_0380_0020.parquet"))
grid3=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=24000.0,log_gravity=3.5,metalicity=0.0,filename="lp0000_24000_0350_0020.parquet"))
grid4=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=24000.0,log_gravity=3.8,metalicity=0.0,filename="lp0000_24000_0380_0020.parquet"))



#### putting it all together

In [ ]:
prof_config=lpd.ProfileConfig(max_velocity=1.0e2,path_to_grids=path_to_grids,wavelength_range=wl_range,intensity_grids=[grid1,grid2,grid3,grid4])

prof_config_dict=prof_config.model_dump(exclude_none=True)

prof_toml_string=tomli_w.dumps(prof_config_dict)
print(prof_toml_string)


### First run

In [ ]:
wavelength_gauss_df = psp.profile_gauss(prof_gauss_toml_string,pulse_df)
wavelength_gauss_df.head(5)

In [ ]:
wavelength_df = psp.profile(prof_toml_string,pulse_df)
wavelength_df.head(5)

# Visualization

### Grayscale plot
Grayscales are one of the most useful ways to asses the variability

In [ ]:
plot_pl.plot_grayscale(wavelength_df=wavelength_df)

### Animation

In [ ]:
#import numpy as np
#from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# plt.rcParams["animation.html"] = "jshtml"
plt.ioff() # To avoid showing the plot two times, you can do plt.ion() afterwards
fig = plt.figure()
ax = plt.axes(xlim=(4098.0, 4102.0), ylim=(0.97, 1.0))
line, = ax.plot([], [], lw=3)
time = plot_pl.extract_time_points_from_data(wavelength_df=wavelength_df)
wavelength = plot_pl.extract_wavelengths(wavelength_df,time=time[0])
def init():
    line.set_data([], [])
    return line,
def animate(i):
    j = i % time.len()
    #print(j)
    #x = wavelength
    #y = plot_pl.extract_flux(wavelength_df=wavelength_df,time=time[j])
    #x, y = wavelength_df.filter(pl.col('time').is_between(j*0.01, (j+1)*0.01)).select(['wavelength', 'normalized flux']).to_numpy().T
    x, y = wavelength_df.filter(pl.col('time').eq(j*0.01)).select(['wavelength', 'normalized flux']).to_numpy().T
    line.set_data(x, y)
    return line,

anim = FuncAnimation(fig, animate, init_func=init, frames=500, interval=20, blit=True); # May take some time
HTML(anim.to_jshtml())

### Actual movement on the surface of the star
If you use triangles for your meshing technique it is possible to see how the surface changes with pulsations

In [ ]:
!ls points*
!ls triangles*
!pwd

First select the file you want to see

In [ ]:
times = [0,0.01]

points_df = pl.read_parquet(f"contents/HackathonPastryLPV/points{times[1]}.parquet")
triangles_df = pl.read_parquet(f"contents/HackathonPastryLPV/triangles{times[1]}.parquet")
points_extracted = points_df[['x coordinate','y coordinate','z coordinate']].to_numpy()
triangles_extracted = triangles_df[['first vertex','second vertex','third vertex']].to_numpy()
faces = np.hstack([np.full((triangles_extracted.shape[0],1),3),triangles_extracted])
mesh = pv.PolyData(points_extracted,faces)
plotter = pv.Plotter(notebook=False, off_screen=True)
actor = plotter.add_mesh(mesh,color='magenta',show_edges=True)
plotter.reset_camera()
plotter.set_position([0,4,-1])
plotter.set_viewup([0,-1,-1])

plotter = pv.Plotter(notebook=True)
plotter.add_mesh(mesh)
plotter.add_axes()
plotter.show(jupyter_backend="html", return_viewer=True)